# AgentCore workshop: building the refund agent

Our company sells electronics online, and our support inbox is drowning in the same
message, over and over: *"I want to refund my order."* Every one of them means a person
looking up the order, checking whether it can be refunded, and replying.

You've been asked to build an **AI refund agent** to take that off the team's plate. Not
a demo: something we can actually put in front of customers. So we build it the way it
grows in real life. Start with the simplest thing that works, then add one AgentCore
feature at a time, each time because the agent ran into something it couldn't handle.

**Everything is done in the AWS console, and there's no code to run.** This notebook tells
the story; the exact steps are the same as in [LABS.md](LABS.md). Where a step says
"paste the contents of a file", the file is in this repo.

| Lab | The question we hit | What you add | AgentCore feature |
|---|---|---|---|
| M1 | Who is the agent talking to? | A login for customers | Cognito → **Identity** (JWT) |
| M2 | Can a customer talk to it at all? | The agent itself | **Harness** on **Runtime** |
| M3 | Does it remember the conversation? | Memory | **Memory** |
| M4 | Can it actually look up an order? | Tools | Lambda + **Gateway** |
| M5 | Can we trust *who* is asking? | A guarded second door | **Gateway** + **Policy** (Cedar, Dogwood) |

**Region for everything: `us-east-1` (N. Virginia).** Set the region picker in the
console before you start. If you build something in the wrong region, nothing will find it.

**Name your resources with your initials.** If your initials are AMG, call your harness
`refund_amg`, your gateway `refund_gw_amg`, and so on. That keeps things tidy, and the
tester uses it to find your resources.

## What we are building

```
   you ---- login ----> Cognito
    |
    |  your token
    |
    +---> Chat tab ---> Harness ---> Gateway A ---> Lambda
    |                   (the agent)   (AWS IAM)      (orders,
    |                                                 refunds)
    |                                                   ^
    +---> Tools tab -----------------> Gateway B -------+
                                       (your token,
                                        policies check
                                        who you are)
```

You won't need all of this at once. By M4 you have the top line: a customer logs in and
talks to the agent, and the agent uses tools. That version works, and it looks secure.
M5 is about why it isn't, and adds the bottom line: a second door into the same tools
that knows exactly who is asking.

## Before we start: the tester app

Our customers will talk to the agent through a website, so we need one too: somewhere
to *be* the customer. A page where you can log in, chat with the agent, and later call
its tools directly. That's the **tester**.

We deploy it the same way we'll deploy the tools in M4: as a **Lambda** with a public
URL. No web server and nothing on your laptop. You set it up once and use it in every
lab.

### What's inside

The tester is one file, `chat/lambda_function.py`, so it can be pasted into the console.
It does three things:

- **Serves the page.** The HTML is at the bottom of the file. In your browser, the page
  logs you in with **Cognito** (M1) and sends your chat messages to the **agent** (M2)
  with your token.
- **Finds your resources: `/lookup`.** It lists the Cognito pools, harnesses and
  gateways in your account and picks yours, so you never copy an ID into it. That's what
  **Find my resources** calls. It only *reads names*, which is all its IAM policy
  (`chat/lambda_policy.json`) allows.
- **Forwards tool calls: `/mcp`.** In M5, the **Tools** tab sends tool calls to
  gateway B through the Lambda, still carrying your own token, so the gateway sees *you*.

> The function URL has no login of its own (auth type **NONE**). That's fine: the page
> holds no secrets, and everything that matters is checked with your Cognito token.

### Deploy it

1. Go to **Lambda** and choose **Create function**.
2. Choose **Author from scratch**. Name it `refund_ui_<your initials>`, runtime
   **Python 3.13**. Choose **Create function**.
3. In the **Code** tab, paste the contents of `chat/lambda_function.py` over the
   default code, then choose **Deploy**.
4. Go to **Configuration**, then **General configuration**, then **Edit**. Set the
   **Timeout** to **15** seconds and save.
5. Go to **Configuration**, then **Permissions**, and click the role name. In the IAM
   page that opens, choose **Add permissions**, then **Create inline policy**, then
   **JSON**. Paste the contents of `chat/lambda_policy.json`, choose **Next**, name it
   `chat-ui-lookup`, and create it.

   > This only lets the app read the names of your pool, harness and gateway, so you
   > never have to copy IDs into it.

6. Back in Lambda, go to **Configuration**, then **Function URL**, then **Create
   function URL**. Choose auth type **NONE** and save.
7. Open the **Function URL** and bookmark it. That is your tester.

As you build things in the labs, they show up under **Settings** in the tester by
themselves. Press **Find my resources** to check again.

**✅ Done when:** the tester page opens and **Find my resources** runs without an error.
Nothing gets a tick yet, because there's nothing to find. That's what the labs fill in.

> If it says *"Lookup failed: AccessDenied"*, the `chat-ui-lookup` policy from step 5
> is missing.

## M1. Identity: who is the agent talking to?

The obvious first move would be to build the agent. But think about what this agent
does: it looks at someone's orders and **moves their money**. Before it can do anything
useful, it has to answer one question every time: *who is this?*

"I'm alice" typed into a chat box doesn't answer that. Anyone can type it. What we
need is a login that hands out a signed **token** saying who you are, one the rest of
the system can check without taking your word for it.

So we start with the thing that issues tokens: an **Amazon Cognito user pool**, with a
few test customers in it.

### Create the user pool

1. Go to **Cognito** and choose **Create user pool**.

2. Under sign-in options, tick **User name**. Leave **Email** unticked.

   > This step matters more than it looks. If you sign in by email, Cognito sets the
   > username to a random ID like `d304d8b2-90e1-70f7`. Later your policies need to
   > compare that username to `alice`, and a random ID will never match. Sign in by
   > user name and the token says `alice`.

3. Cognito will now ask you to pick a required attribute. Choose **email**. That is
   fine. A required attribute is not the same as a sign-in option, so your username
   stays `alice`.

4. Turn **self-registration off**. You will add the users yourself.

5. Create an app client:
   - Type: **Single-page application**, so it has no client secret.
   - Name it `refund-tester-<your initials>`.

6. Name the pool `refund-pool-<your initials>` and create it.

7. Open the app client and turn on **ALLOW_USER_PASSWORD_AUTH** under authentication
   flows. The tester app needs it.

### Add your customers

8. Add three users. For each one choose **Don't send an invitation**, tick **Mark email
   address as verified**, and set the password to `Workshop#2026`.

   | User name | Email |
   |---|---|
   | `alice` | `alice@example.com` |
   | `bob` | `bob@example.com` |
   | `mateo` | `mateo@example.com` |

   These are the customers for the rest of the workshop. Each one has their own
   orders, and the whole point is that they should only ever see their own.

### Check the tester can see it

9. Open the tester and press **Find my resources**. **User pool** and **App client**
   get a tick.

**✅ Done when:** **User pool** and **App client** both have a tick.

You can't log in just yet: the tester only opens the login once there is an agent to
talk to. The login is ready and waiting; in M2 we give it something to protect.

## M2. The agent: can a customer talk to it?

Now we have customers who can prove who they are. Time to give them something to talk
to.

We start with the smallest thing that could possibly help: an agent with a clear job
description, running somewhere real. No database, no tools yet. We want it **deployed**,
not on a laptop, because that's where it has to live eventually.

A **Harness** is a declarative agent: you pick a model, write a system prompt, and attach
tools, and AgentCore runs it on **Runtime** for you. No code, no container. And because
we already have a login, we can put it on the front door from day one: only someone with
a valid token from *our* pool gets in.

### Create the harness

1. Go to **Bedrock AgentCore** and then **Harness**. Choose **Create**.

2. Name it `refund_<your initials>`. The name cannot be changed later.

3. Pick a Claude model that is available in `us-east-1`.

4. For the system prompt, paste:

   ```
   You are a customer-service refund assistant. Help the signed-in customer with
   their orders and refunds. Be brief.
   ```

5. Set **Inbound auth** to **Custom JWT**. This is where M1 plugs in:

   - **Discovery URL**, with your own pool ID in it:
     ```
     https://cognito-idp.us-east-1.amazonaws.com/YOUR_POOL_ID/.well-known/openid-configuration
     ```
   - **Allowed clients**: your app client ID.

6. Save and wait for the status to become **READY**.

7. Copy the **harness ARN** from the details page. It looks like
   `arn:aws:bedrock-agentcore:us-east-1:123456789012:harness/refund_amg-AbC1234567`.

   > Take the ARN with `:harness/` in it. There is a second ARN on the page with
   > `:runtime/harness_` in it. That one will not work.

8. In the tester, press **Find my resources**. **Agent (harness)** gets a tick, and
   the login form appears.

### Log in and talk to it

9. Log in as **alice**. The first login asks you to pick a new password.

10. Open the panel **What's inside my token?**. It should show `"username": "alice"`.
    If it shows a long random ID instead, your pool signs in by email: go back to M1
    step 2.

    > 🔎 Look at that panel for a moment. `"username": "alice"` is not something you
    > typed into a chat; Cognito signed it, and the agent only let you in because the
    > signature checks out. Keep that difference in mind. By the end of the workshop,
    > it's the difference between an agent that *looks* safe and one that *is*.

11. Send **hello** in the tester's Chat tab.

12. Now open your harness in the console and choose **Test Harness**. Ask the same thing
    there. It works without any login, because the console signs you in with your AWS
    user. In the playground you can type any **Actor ID** you like, and the agent
    believes it.

**✅ Done when:** you are signed in as alice, your token says `"username": "alice"`, and
the agent replies.

> 🔎 Try asking it: *"I want to refund my keyboard."* It's polite, it understands what
> you want, and it can't do a single useful thing about it: it has no way to look an
> order up. If it claims it found your order, it's guessing. Tools fix that in M4.
>
> And notice step 12: in the playground, *you* chose who the agent thinks it's talking
> to. The rest of the labs replace that guess with a checked identity.

## M3. Memory: does it remember the conversation?

Picture a real customer. They explain their problem, go and find the order number, and
come back a minute later. If the agent greets them with *"Hi! How can I help?"*, they
have to start again, and they'll be annoyed before we've done anything.

An agent doesn't remember anything by itself: each message is a new call to the model.
**AgentCore Memory** keeps it, in two layers:

- **Short-term memory:** the conversation so far, for one chat (one *session*).
- **Long-term memory:** facts pulled *out* of conversations and kept for the
  customer, so a new chat can still use them.

Your harness already has both. When you created it, AgentCore gave it a **managed
memory** with long-term strategies. To see what each layer does, we'll take the
long-term layer away first, then put it back.

### Look at what you have

1. Open your harness and find its **Memory** section. Note the two retrieval configs:
   - `/actors/{actorId}/facts/` (semantic): facts about **you**, kept across chats.
   - `/actors/{actorId}/summaries/{sessionId}/` (summarization): a summary of
     **one chat**.

   `{actorId}` is your login. The tester sends your username, so alice's memory
   and bob's memory never mix.

### Short-term memory only

2. Go to **AgentCore** and then **Memory**. Create a memory resource in
   `us-east-1`. Name it `refund_memory_<your initials>`. **Do not add any
   long-term strategies.**

3. Edit your harness and switch its memory to the one you just made. Wait for
   **READY**.

4. In the Chat tab, say: **my favourite colour is green.**

5. Then ask: **what is my favourite colour?** It remembers. That is short-term
   memory: the conversation so far, kept for this chat.

6. Click **New chat** and ask again. It has forgotten. A new chat is a new
   session, and short-term memory belongs to the session.

This is the customer from the start of this section: they come back, and the agent asks
them to start again.

### Long-term memory

7. Edit your harness and switch its memory back to **managed memory**. Wait for
   **READY**, and check the two retrieval configs from step 1 are there again.

8. In the Chat tab, say: **my favourite colour is green.** Ask about it in the same
   chat. It remembers, as before.

9. **Wait a minute or two.** Long-term memories are extracted in the background
   after the conversation, not instantly.

10. Click **New chat** and ask: **what is my favourite colour?** This time it
    remembers. The fact was saved under `/actors/alice/facts/`, which belongs to
    alice, not to the chat.

11. Sign out, log in as **bob**, and ask the same question. bob's agent does not
    know. Same agent, same memory resource, different actor.

**✅ Done when:** the plain memory forgets after **New chat**, the managed memory
remembers after **New chat**, and bob does not see alice's colour.

> If step 10 still forgets, wait another minute and ask again. Extraction can take a
> little while.

> 🔎 Step 11 is the M1 login paying off again: memory is kept per user because the actor
> ID comes from your token. And step 9 is worth a second thought. Long-term memory is
> *eventually* up to date, not instantly. For a favourite colour, that's fine. Now
> imagine a customer halfway through a refund: should the agent trust what it
> *remembers* about the refund's status, or ask the order system every time?

## M4. Tools: can it actually look up an order?

So far our agent can log you in, talk, and remember. Ask it about your keyboard and the
best it can say is still *"I can't look that up."* Or it will make up some numbers.

To help with a refund, the agent has to do what a support person does first: **look the
order up**. Which order does the customer mean? Has it been delivered? What was paid, and
has it already been refunded? And then, if everything checks out, actually **issue the
refund**. All of that lives in our order system.

So we give the agent **tools**: functions it can decide to call when it needs a fact it
doesn't have, or needs to do something.

### The tools: a Lambda

The tools live in one Lambda function, `tools/lambda_function.py`. For now the "order
system" is a set of fake orders inside it. Later it could be a real database, and the
agent wouldn't notice the difference.

- **Four tools:** `find_orders` (for "refund my keyboard"), `get_order_transaction`
  (status, payment, existing refund), `process_refund` (simulated, no real money), and
  `get_refund_status`.
- **Every tool takes a `customer_id`**, and only answers about that customer's orders.
- **The Lambda never checks who is really calling.** It trusts the `customer_id` it is
  given. Remember that; it's the whole story of M5.

1. Go to **Lambda** and choose **Create function**.
2. Name it `refund_tool_<your initials>`, runtime **Python 3.12** or newer.
3. Paste the contents of `tools/lambda_function.py` over the default code, then choose
   **Deploy**.
4. Copy the function ARN.

The Lambda holds the fake orders. alice owns A-1001 to A-1004, bob owns B-2001 and
B-2002, mateo owns M-3001 and M-3002.

### The front door: gateway A

We could wire the Lambda straight into the agent. But this won't stay one Lambda: next
come shipping, maybe a CRM, and other agents will want the same tools. Instead of every
agent wiring up every tool itself, we put the tools behind one front door that handles
discovery, auth and routing. That front door is the **AgentCore Gateway**. It turns the
Lambda into an **MCP server** any agent can connect to.

To do that, the gateway needs a **tool schema** (`tools/tool-schema.json`). It's worth a
minute, because the schema is read by the **model**, not by the Lambda. For each tool
it says:

- **`name`**: what the agent calls. It must match a tool the Lambda knows.
- **`description`**: *when* to use it. The model chooses tools by reading this, so it's
  effectively part of your prompt.
- **`inputSchema`**: the arguments, and which ones are required.

5. Go to **AgentCore**, then **Gateways**, then **Create gateway**.
6. Name it `refund_gw_<your initials>`. Protocol: **MCP**.
7. Set inbound auth to **AWS IAM**. This is the agent's door, so the caller is the
   agent, not a person.
8. For permissions, let it create a new service role.
9. Add a target:
   - Target name: `orders`
   - Target type: **Lambda**, pointing at the function you just made
   - Tool schema: paste `tools/tool-schema.json`
   - Outbound auth: the gateway IAM role
10. Create it and wait for **READY**.

> If you get *"Gateway service is not authorized to perform AssumeRole on Gateway
> role"*, wait a minute and add the target again. The new role takes a moment to become
> usable.

### Hook it up

11. Edit your harness, go to **Tools**, and add an **AgentCore Gateway** tool pointing at
    your gateway ARN. Outbound auth: **AWS IAM**. Wait for **READY**.

12. The agent now needs to know who it is talking to, because every tool wants a
    `customer_id`. In the tester's **Settings**, tick **Tell the agent who I am**.

    The tester now adds a line to every message saying who is signed in, filled in from
    your token. Open **Settings**, then **Enter manually** to read it.

13. Log in as alice and ask: **what orders do I have?**

**✅ Done when:** alice sees her four orders and the Lambda's CloudWatch log shows the
`find_orders` call.

Now go back to the customer from M2, the one with the keyboard, and help them properly
this time. A customer describes their order in their own words, and the agent finds it,
checks it, and gives a real answer from real data instead of guesses. That's the first
version we could actually show a customer.

Which is exactly when the support team starts trying to break it.

### Now look at what you just built

Ask the agent: **show me order B-2001.** That is bob's order, and you get nothing back.

It feels secure. It is not.

Go back to step 12. The agent knows you're alice because the tester **typed alice's name
into the prompt**, and the model chose to obey. Nothing checked it. Remember M1: the
token says alice and Cognito signed it, but by the time it reaches the tools it's just a
sentence. A cleverer message can talk the model into passing `customer_id` of `bob`,
and the Lambda, which trusts whatever `customer_id` it gets, will happily answer.

And the gateway can't help here. When the agent calls it, the gateway sees the
**agent's** AWS role. It never sees you.

That's the production question this whole workshop is building towards: **a valid
login, an unauthorized refund.** M5 fixes it.

## M5. Policies: can we trust who is asking?

The fix isn't a better prompt. A prompt is a request; we need a rule the model can't
argue with, checked *outside* the model, against an identity it can't make up.

So we build a second door into the same tools, one that does see you, and we put a
guard on it.

- **Gateway B** takes your **token** instead of the agent's AWS role. Now the gateway
  knows the caller is alice, because Cognito signed it.
- A **policy engine** sits on gateway B and checks **every tool call** against rules
  written in **Cedar**: *is this person allowed to make this call, with these
  arguments?*

You'll call gateway B from the tester's **Tools** tab: the tool calls the agent would
make, but sent with your own login.

### Create gateway B

1. Create another gateway, named `refund_gw_jwt_<your initials>`. Protocol: **MCP**.

2. This time set inbound auth to **Custom JWT**, with the same two values you used for
   the harness in M2:
   - Discovery URL with your pool ID
   - Allowed clients: your app client ID

3. Add a target. Use the **same Lambda** and the **same tool schema** as M4.

   > Name this target `orders` exactly. Policy rules refer to tools as
   > `<target name>___<tool name>`, so the rules below only work if the target is
   > called `orders`.

4. Give gateway B permission to keep a policy session. Open gateway B in **AgentCore**
   and click its **service role**. In IAM choose **Add permissions**, then **Create
   inline policy**, then **JSON**. Paste the contents of
   `policies/gateway_temporal_iam.json`, choose **Next**, name it `policy-sessions`, and
   create it.

   > The tester tags every Tools call with a session, so the policies can see what
   > happened earlier. Without this permission, every allowed call fails with
   > `Failed to get workload identity token` once a policy engine is attached.

5. Copy the gateway's **ARN** and its **URL**. The URL ends in `/mcp`.

6. In the tester, press **Find my resources**. **Gateway (Tools tab)** gets a tick, and
   the **Tools** tab starts working.

7. In the Tools tab, call `orders___find_orders` with `customer_id` set to `alice`. It
   works. Now set it to `bob`. **It also works.** The gateway knows you're alice, but
   nothing is using that yet. There is no guard.

### Add the guard

8. Go to **AgentCore**, then **Policy**, and create a policy engine named
   `refund_identity_<your initials>`. Use letters, digits and underscores only. No
   hyphens.

9. Add a policy called `identity_binding`. Replace `YOUR_GATEWAY_B_ARN` with the ARN
   from step 5.

   ```cedar
   permit (
       principal is AgentCore::OAuthUser,
       action in [
           AgentCore::Action::"orders___find_orders",
           AgentCore::Action::"orders___get_order_transaction",
           AgentCore::Action::"orders___process_refund",
           AgentCore::Action::"orders___get_refund_status"
       ],
       resource == AgentCore::Gateway::"YOUR_GATEWAY_B_ARN"
   )
   when {
       principal.hasTag("username") &&
       context.input.customer_id == principal.getTag("username")
   };
   ```

   This says: you may use these four tools, but only when the `customer_id` you send
   matches the user name on your token. This is where the `"username": "alice"` from
   M1 finally does its job.

10. Add a second policy called `refund_cap_200`, same ARN:

    ```cedar
    forbid (
        principal is AgentCore::OAuthUser,
        action == AgentCore::Action::"orders___process_refund",
        resource == AgentCore::Gateway::"YOUR_GATEWAY_B_ARN"
    )
    when {
        context.input.amount.greaterThan(decimal("200.0"))
    };
    ```

    This says: never pay out more than $200, whoever asks. A `forbid` always beats a
    `permit`.

11. Attach the policy engine to gateway B and set the mode to **LOG_ONLY** first. In
    this mode it writes down what it would have done but still lets everything through.
    Make a few calls, then switch it to **ENFORCE**.

    > Rolling out a guard in log-only mode first is how you'd do it in production too:
    > see what it *would* block before it blocks real customers.

### Show that it works

Log in as **alice** and use the Tools tab:

| What you call | What should happen |
|---|---|
| `find_orders` with `customer_id` = `alice` | allowed |
| `find_orders` with `customer_id` = `bob` | denied |
| `get_order_transaction` with `customer_id` = `bob`, `order_id` = `B-2001` | denied |
| `process_refund`, `alice`, `A-1004`, amount `300` | denied, over the cap |
| `process_refund`, `alice`, `A-1001`, amount `49` | allowed |

**✅ Done when:** you can show one allow and the three denies.

### Add a rule with memory (Dogwood)

The support team has one more worry. A refund should never be the *first* thing that
happens to an order: someone should have looked at it first. But Cedar judges each call
on its own. It cannot know whether you looked at an order before you asked to refund it.

**Dogwood** is Cedar plus the history of your session, so it can.

12. In your policy engine, add a policy called `refund_after_lookup`. Choose **Dogwood**
    as the policy language, and replace `YOUR_GATEWAY_B_ARN` again:

    ```
    forbid (
        principal is AgentCore::OAuthUser,
        action == AgentCore::Action::"orders___process_refund",
        resource == AgentCore::Gateway::"YOUR_GATEWAY_B_ARN"
    )
    unless temporal {
        formerly within 1h AgentCore::Action::"orders___get_order_transaction"::response{
            eventResource: resource,
            input.customer_id: context.input.customer_id,
            input.order_id: context.input.order_id
        }
    };
    ```

    This says: no refund unless, earlier in this session, the same customer opened the
    same order. Like the cap, it is a `forbid` on top of the first policy.

    > From now on every call must say which session it belongs to. The tester does that
    > for you: the Tools tab shows your **Policy session**.

### Show that it remembers

Log in as **bob**, open the Tools tab, and click **New session** first.

| What you call, in order | What should happen |
|---|---|
| `process_refund`, `bob`, `B-2001`, amount `15` | denied, you never looked at it |
| `get_order_transaction`, `bob`, `B-2001` | allowed |
| `process_refund`, `bob`, `B-2001`, amount `15` | allowed |

Wait a second between the last two: the lookup is recorded just after it returns. Then
click **New session** and try the refund again. It is denied, because the new session
has no history. (The policy decides first, so you see the deny, not the Lambda saying
the order is already refunded.)

**✅ Done when:** the same refund is denied, then allowed, then denied again in a new
session.

## Where we are

Look back at the path:

- **M1** gave every customer a signed identity.
- **M2** put an agent behind that login.
- **M3** let it remember a conversation, and showed where that memory ends.
- **M4** gave it tools, and it started actually helping. It also *looked* secure.
- **M5** showed why it wasn't, and fixed it.

Two doors into the same Lambda.

Through the agent, your name is a sentence in a prompt. The model usually follows it,
but nothing makes it.

Through the Tools tab, your name is a signed token the gateway checked, and the policy
engine compares it to every tool call. The model cannot argue with it and neither can
you.

That is the difference between telling software who you are and proving it. Cedar
checks who you are on every call. Dogwood also checks what you did before it.

If something breaks along the way, ask a staff member. Most problems are listed in
[STAFF.md](STAFF.md), and the tester shows a hint under most errors.